# ECE 491 - Python Lab 2
## Basic Image Processing Using Python — Upload-Image Version

This notebook follows the professor's Lab 2 handout in order.

**Important:** this version does **not** recreate, extract, encode, or embed the Boat or Barbara input images.

You will upload the original professor-provided image files directly in Google Colab:

1. upload the **Boat** image as `.png`, `.jpg`, or `.jpeg`;
2. complete the grayscale Boat exercises;
3. upload the **Barbara** image as `.png`, `.jpg`, or `.jpeg`;
4. complete the RGB filtering exercises.

The notebook covers:

1. normalized Boat pixel value at `(132,112)`;
2. top-left quarter using 2-D slicing;
3. 3-by-3 moving-average filtering with `scipy.signal.convolve2d`;
4. saving the filtered image;
5. the specified 3-by-3 difference filter;
6. the handout's horizontal-edge experiment using `[-1, 2, -1]`;
7. the handout's vertical-edge experiment using the transposed filter;
8. repeating the filtering experiments on Barbara by filtering the R, G, and B planes independently.

> Use the original images supplied by the professor.  
> All displayed mathematics uses `$$ ... $$`.


## How to run this notebook in Google Colab

1. Run the Boat upload cell and select the original **Boat** `.png`, `.jpg`, or `.jpeg`.
2. Run the Boat analysis cells from top to bottom.
3. When you reach the Barbara section, run its upload cell and select the original **Barbara** image.
4. Continue through the remaining cells.
5. Colab will generate only the **processed output images** required by the lab; it does not recreate the two input images.


In [ ]:
# Upload the professor-provided BOAT image.
# Accepted formats: PNG, JPG, JPEG.

from google.colab import files
from pathlib import Path

print("Please upload the BOAT image now (.png, .jpg, or .jpeg).")
boat_upload = files.upload()

if len(boat_upload) != 1:
    raise ValueError("Please upload exactly ONE Boat image in this cell.")

BOAT_FILE = next(iter(boat_upload.keys()))
boat_ext = Path(BOAT_FILE).suffix.lower()

if boat_ext not in {".png", ".jpg", ".jpeg"}:
    raise ValueError("Boat image must be PNG, JPG, or JPEG.")

print("Boat image uploaded:", BOAT_FILE)


In [ ]:
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt
from scipy.signal import convolve2d


def show_gray(arr, title, shift=0.0):
    plt.figure(figsize=(6,6))
    plt.imshow(np.clip(arr + shift, 0, 1), cmap="gray", vmin=0, vmax=1)
    plt.title(title)
    plt.axis("off")
    plt.show()


def show_rgb(arr, title, shift=0.0):
    plt.figure(figsize=(6,6))
    plt.imshow(np.clip(arr + shift, 0, 1))
    plt.title(title)
    plt.axis("off")
    plt.show()


def filter_rgb_full(img_rgb, kernel):
    return np.stack([convolve2d(img_rgb[:,:,c], kernel) for c in range(3)], axis=2)

# Boat image setup

The handout first reads the Boat image in grayscale and converts it from `uint8` to `float32` normalized by 255.

This notebook reads the file that you uploaded in the previous cell.


In [ ]:
img_u8 = cv.imread(BOAT_FILE, cv.IMREAD_GRAYSCALE)

if img_u8 is None:
    raise ValueError(f"OpenCV could not read the uploaded Boat image: {BOAT_FILE}")

print("Boat filename:", BOAT_FILE)
print("Shape:", img_u8.shape)
print("dtype before normalization:", img_u8.dtype)

img = img_u8.astype(np.float32) / 255.0

print("dtype after normalization:", img.dtype)
print("Normalized range:", float(img.min()), "to", float(img.max()))

show_gray(img, "Original Boat image")


# 1) Pixel at location `(132,112)`

With NumPy image indexing, `(132,112)` means row 132, column 112.

For the appendix Boat image, the original 8-bit pixel is 158. After normalization:

$$
\frac{158}{255}=0.619607843\ldots
$$

Because the array is converted to `float32`, Python reports approximately:

$$
\boxed{0.6196078658}
$$

In [ ]:
print("uint8 pixel =", img_u8[132,112])
print("normalized pixel =", float(img[132,112]))

# 2) Top-left quarter

For the professor's Boat image, the size is 512-by-512, so half of each dimension is 256.

More generally, if the uploaded image has height $H$ and width $W$, the top-left quarter is

$$
\text{rows }0:\frac{H}{2},
\qquad
\text{columns }0:\frac{W}{2}.
$$

For the original 512-by-512 Boat image this becomes

$$
\boxed{a=0,\quad b=256}
$$

and

```python
img[0:256, 0:256]
```


In [ ]:
H, W = img.shape
half_h = H // 2
half_w = W // 2

quarter = img[0:half_h, 0:half_w]

print("Original shape:", img.shape)
print("Quarter indices: rows 0:", half_h, ", columns 0:", half_w, sep="")
print("Quarter shape:", quarter.shape)

show_gray(quarter, "Boat - top-left quarter")


# 3) 3-by-3 moving-average box filter

The professor defines

$$
M_a=\frac19
\begin{bmatrix}
1&1&1\\
1&1&1\\
1&1&1
\end{bmatrix}.
$$

`scipy.signal.convolve2d(img, ma)` uses `mode='full'` by default.

For an image of size

$$
H\times W
$$

and a 3-by-3 filter, the full-convolution output size is

$$
(H+3-1)\times(W+3-1).
$$

For the professor's 512-by-512 Boat image:

$$
\boxed{514\times514}.
$$

Visually, the moving average acts as a low-pass filter: the image becomes smoother, sharp edges are softened, and fine texture is reduced.


In [ ]:
ma = (1./9.)*np.array([[1.,1.,1.],[1.,1.,1.],[1.,1.,1.]])
img_filtered = convolve2d(img, ma)
print("Filtered size:", img_filtered.shape)
show_gray(img_filtered, "Boat - 3x3 moving-average filter")

# 4) Save the filtered image

The filtered image is floating point in approximately `[0,1]`. For reliable PNG saving with OpenCV, convert it to 8-bit after clipping and scaling by 255.

In [ ]:
saved = np.round(np.clip(img_filtered,0,1)*255).astype(np.uint8)
cv.imwrite("boat_moving_average.png", saved)
print("Saved boat_moving_average.png")

# 5) 3-by-3 difference filter

Use exactly the matrix stated by the professor:

$$
\boxed{
\begin{bmatrix}
1&0&-1\\
2&0&-2\\
1&0&-1
\end{bmatrix}}
$$

The full-convolution output is again 514-by-514. The filter produces strong positive/negative responses around intensity transitions while relatively uniform regions are closer to zero. Following the handout, add 0.5 before display; clipping is used only to keep the displayed image in the legal `[0,1]` range.

In [ ]:
difference_filter = np.array([[1.,0.,-1.],[2.,0.,-2.],[1.,0.,-1.]])
print(difference_filter)
img_difference = convolve2d(img, difference_filter)
print("Difference output shape:", img_difference.shape)
print("Raw range:", img_difference.min(), img_difference.max())
show_gray(img_difference, "Boat - difference filter (+0.5)", shift=0.5)

# 6) Horizontal-edge experiment

Following the professor's wording exactly, use the row filter

$$
\boxed{[-1\quad 2\quad -1]}.
$$

With a 512-by-512 image and a 1-by-3 filter, full convolution produces

$$
\boxed{512\times514}.
$$

In [ ]:
horizontal_filter = np.array([[-1.,2.,-1.]])
boat_horizontal = convolve2d(img, horizontal_filter)
print("Horizontal-edge output shape:", boat_horizontal.shape)
show_gray(boat_horizontal, "Boat - horizontal-edge experiment (+0.5)", shift=0.5)

# 7) Vertical-edge experiment

Use the transpose:

$$
\boxed{
\begin{bmatrix}-1\\2\\-1\end{bmatrix}}
$$

The full-convolution output size is

$$
\boxed{514\times512}.
$$

In [ ]:
vertical_filter = horizontal_filter.T
boat_vertical = convolve2d(img, vertical_filter)
print("Vertical-edge output shape:", boat_vertical.shape)
show_gray(boat_vertical, "Boat - vertical-edge experiment (+0.5)", shift=0.5)

# 8) Repeat with color Barbara

Now upload the professor-provided **Barbara** image directly.

Accepted formats are `.png`, `.jpg`, and `.jpeg`.

OpenCV reads a color image as **BGR**, so after loading it we convert BGR to RGB before display and filtering. Each R, G, and B plane is filtered independently, exactly as requested by the handout.


In [ ]:
# Upload the professor-provided BARBARA image.
# Accepted formats: PNG, JPG, JPEG.

from google.colab import files
from pathlib import Path

print("Please upload the BARBARA image now (.png, .jpg, or .jpeg).")
barbara_upload = files.upload()

if len(barbara_upload) != 1:
    raise ValueError("Please upload exactly ONE Barbara image in this cell.")

BARBARA_FILE = next(iter(barbara_upload.keys()))
barbara_ext = Path(BARBARA_FILE).suffix.lower()

if barbara_ext not in {".png", ".jpg", ".jpeg"}:
    raise ValueError("Barbara image must be PNG, JPG, or JPEG.")

barbara_bgr = cv.imread(BARBARA_FILE, cv.IMREAD_COLOR)

if barbara_bgr is None:
    raise ValueError(f"OpenCV could not read the uploaded Barbara image: {BARBARA_FILE}")

barbara_rgb = cv.cvtColor(barbara_bgr, cv.COLOR_BGR2RGB)
barbara = barbara_rgb.astype(np.float32) / 255.0

print("Barbara filename:", BARBARA_FILE)
print("Barbara shape:", barbara.shape)
print("Barbara dtype after normalization:", barbara.dtype)

show_rgb(barbara, "Original Barbara - RGB")


In [ ]:
barbara_ma = filter_rgb_full(barbara, ma)
print("Barbara moving-average shape:", barbara_ma.shape)
show_rgb(barbara_ma, "Barbara - moving-average filter")

# Convert RGB -> BGR before cv.imwrite.
barbara_ma_u8 = np.round(np.clip(barbara_ma,0,1)*255).astype(np.uint8)
barbara_ma_bgr = cv.cvtColor(barbara_ma_u8, cv.COLOR_RGB2BGR)
cv.imwrite("barbara_moving_average.png", barbara_ma_bgr)

In [ ]:
barbara_difference = filter_rgb_full(barbara, difference_filter)
print("Barbara difference shape:", barbara_difference.shape)
show_rgb(barbara_difference, "Barbara - difference filter (+0.5)", shift=0.5)

barbara_horizontal = filter_rgb_full(barbara, horizontal_filter)
print("Barbara horizontal-edge shape:", barbara_horizontal.shape)
show_rgb(barbara_horizontal, "Barbara - horizontal-edge experiment (+0.5)", shift=0.5)

barbara_vertical = filter_rgb_full(barbara, vertical_filter)
print("Barbara vertical-edge shape:", barbara_vertical.shape)
show_rgb(barbara_vertical, "Barbara - vertical-edge experiment (+0.5)", shift=0.5)

# Final answers / cheat sheet

- **Input workflow:** upload the original professor-provided Boat and Barbara image files directly in Colab.
- Accepted input formats in this notebook: **PNG, JPG, JPEG**.
- No image is recreated, extracted, Base64-decoded, or embedded by the notebook.
- For the professor's Boat image, size: **512-by-512**.
- Normalized Boat pixel `(132,112)`: **0.6196078658** (`158/255`) when using the original appendix image.
- Top-left quarter for a 512-by-512 image: `img[0:256,0:256]`.
- A 3-by-3 **full** convolution of a 512-by-512 image gives **514-by-514**.
- Moving average: **low-pass smoothing**; edges and fine details are softened.
- 3-by-3 difference filter: highlights intensity transitions; add **0.5** before display as requested.
- Row `[-1,2,-1]` full output for 512-by-512 input: **512-by-514**.
- Column `[-1,2,-1]^T` full output: **514-by-512**.
- Barbara must be converted **BGR -> RGB** before display/processing.
- Filter Barbara by applying `convolve2d` independently to the **R, G, and B planes**.
- Convert **RGB -> BGR** before saving a color image with `cv.imwrite`.
